# RSNA Knee — MedGemma Teacher Feasibility and LoRA Pilot

**Private research notebook · 13 September 2026**

Teacher candidate: **`google/medgemma-1.5-4b-it`**. The intended sequence is to audit the data, adapt a medical teacher, measure its predictions on held-out studies, then distill a smaller image-only student. This notebook implements the teacher pilot and score export. Student training is a subsequent experiment after the teacher proves useful.

**Run All defaults to a CPU data audit and MRI preprocessing check. It does not download or train MedGemma.** Set `RUN_MEDGEMMA=True` only after arranging model access and suitable GPU compute. No competition submission or public upload is performed.

Why this model: MedGemma 1.5 supports MRI volume representations and task adaptation. Meta fastMRI's published reconstruction models solve a different problem: recovering images from undersampled scanner measurements. Kaggle supplies reconstructed images and asks for 12 abnormality scores. fastMRI is therefore not our first classification teacher.

**Hardware:** Google's official fine-tuning tutorial specifies BF16 support and at least 40 GB GPU memory. A Kaggle T4 has 16 GB and does not support native BF16; two T4s do not automatically combine memory. This notebook conservatively requires one GPU with BF16 and roughly 40 GB memory for its optional QLoRA pilot. Six images per study may still require adjustment or a larger GPU. A100 80 GB gives more room; no paid instance is started here. An optimized T4 adaptation remains untested.

**Limits:** Only a small subset has organizer labels. This pilot uses those labels, never treats missing values as negative, and never provides the report to the image teacher as input. It samples six slices rather than a complete MRI volume, so this is a memory/training experiment, not a final competitive architecture or validation result.

**Access:** Obtain access to the official model after reviewing Google's HAI-DEF terms. On Kaggle add a read-only token as the `HF_TOKEN` secret and grant this notebook access. Do not paste tokens into cells. This notebook does not accept model terms on your behalf. HAI-DEF obligations also apply to models distilled to imitate it.


## 1. Audit setup
Retain the Kaggle PyTorch stack. The CPU audit needs DICOM decoders; Internet must be enabled to install them. The optional model stage installs the Hugging Face libraries separately. To run outside Kaggle, set `DATA_DIR` to your authorized competition data and `OUTPUT_DIR` to a private output directory.


In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable,'-m','pip','install','--quiet',
                       'pydicom>=3,<4','python-gdcm>=3,<4'])
import os, json, time, hashlib, random, gc
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import pydicom
from pydicom.pixels import apply_modality_lut
import torch
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score
from tqdm.auto import tqdm

RUN_MEDGEMMA = False
MODEL_ID = 'google/medgemma-1.5-4b-it'
DATA_DIR = ''
OUTPUT_DIR = '/kaggle/working/medgemma_knee'
SEED = 42
MAX_LABELED_STUDIES = 80
SLICES_PER_PLANE = 2
HOLDOUT_FRACTION = 0.25
MAX_STEPS = 20                 # feasibility test, not full training
ACCUMULATION = 4
LORA_RANK = 8
LEARNING_RATE = 5e-5
LABELS = ['ACL','MCL','Medial Meniscus','Lateral Meniscus','Medial OA',
          'Lateral OA','PF OA','Effusion','Synovitis',"Baker's",'Contusion','Fracture']
PLANES = ['Sagittal','Coronal','Axial']
OUT=Path(OUTPUT_DIR); OUT.mkdir(parents=True,exist_ok=True)
TEMP=Path('/kaggle/temp/medgemma_knee') if Path('/kaggle').exists() else OUT/'cache'
TEMP.mkdir(parents=True,exist_ok=True)
os.environ.setdefault('HF_HOME',str(TEMP/'huggingface'))
os.environ['TOKENIZERS_PARALLELISM']='false'
os.environ['HF_HUB_DISABLE_TELEMETRY']='1'
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print('Mode:', 'MedGemma pilot requested' if RUN_MEDGEMMA else 'CPU audit; no model training')
print('GPU devices:',[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
def log_metric_event(step,values):
    with (OUT/'metrics.jsonl').open('a') as f:
        f.write(json.dumps(dict(step=step,values=values))+'\n')


## 2. Audit the real labels
Read only the CSVs first. Do not recursively list hundreds of thousands of DICOM files. Original reports are kept in memory for duplicate grouping, and are not printed or exported. The organizer labels take precedence over any future report-derived labels.


In [ ]:
candidates = [Path('/kaggle/input/competitions/rsna-knee-abnormality-detection'),
              Path('/kaggle/input/rsna-knee-abnormality-detection')]
if DATA_DIR: candidates.insert(0,Path(DATA_DIR))
DATA = next((p for p in candidates if (p/'train.csv').exists()), None)
assert DATA is not None, 'Attach the RSNA Knee Abnormality Detection competition using Add Input.'
train = pd.read_csv(DATA/'train.csv', dtype={'StudyInstanceUID':str})
series = pd.read_csv(DATA/'train_series.csv', dtype={'StudyInstanceUID':str, 'SeriesInstanceUID':str})
test = pd.read_csv(DATA/'test.csv', dtype={'StudyInstanceUID':str})
test_series = pd.read_csv(DATA/'test_series.csv', dtype={'StudyInstanceUID':str, 'SeriesInstanceUID':str})
sample = pd.read_csv(DATA/'sample_submission.csv', dtype={'StudyInstanceUID':str})
assert list(sample.columns) == ['StudyInstanceUID']+LABELS, 'Unexpected submission schema; review labels.'
assert train.StudyInstanceUID.is_unique and test.StudyInstanceUID.is_unique
assert set(train.StudyInstanceUID).isdisjoint(test.StudyInstanceUID)
assert set(series.StudyInstanceUID).issubset(set(train.StudyInstanceUID))
for label in LABELS:
    train[label] = pd.to_numeric(train[label], errors='raise')
    assert train[label].dropna().isin([0,1]).all(), f'Non-binary organizer labels: {label}'
audit = pd.DataFrame({'known':train[LABELS].notna().sum(),
                      'positive':(train[LABELS]==1).sum(),
                      'negative':(train[LABELS]==0).sum(),
                      'missing':train[LABELS].isna().sum()})
display(audit)
audit.to_csv(OUT/'label_audit.csv', index_label='label')
available = train[train[LABELS].notna().any(axis=1)].copy()
pilot = available.sample(n=min(len(available),MAX_LABELED_STUDIES), random_state=SEED).reset_index(drop=True)
assert len(pilot)>=12, 'Too few organizer-labeled studies for this pilot; add audited labels first.'
print(f'Training studies: {len(train):,}; series: {len(series):,}')
print(f'Organizer-labeled studies: {len(available)}; pilot: {len(pilot)}; visible test: {len(test)}')
print('Missing labels remain unknown. No report-derived labels are used in this version.')


## 3. MRI decoding and geometry
Select one series per plane, preferring fluid-sensitive/fat-suppressed acquisitions. Sort slices by position projected onto the DICOM slice normal, with InstanceNumber as a logged fallback. Use dominant-axis flips/transposition for consistent in-plane orientation, preserve physical aspect ratio using PixelSpacing, and pad instead of tightly cropping the knee. This is a 2D slice pilot, not full 3D registration. Unsupported/corrupt studies are reported and excluded, never silently assigned blank images.


In [ ]:
ISSUES = []
HEADER_TAGS = ['ImagePositionPatient','ImageOrientationPatient','InstanceNumber',
               'PixelSpacing','PatientID','Manufacturer','ManufacturerModelName','MagneticFieldStrength']

def header(path):
    return pydicom.dcmread(path, stop_before_pixels=True, specific_tags=HEADER_TAGS)

def rank_series(rows):
    rows = rows.copy()
    rows['_rank'] = (pd.to_numeric(rows['Fluid_Sensitive'], errors='coerce').fillna(0)*2
                     + pd.to_numeric(rows['Fat_Suppression'], errors='coerce').fillna(0))
    return rows.sort_values(['_rank','SeriesInstanceUID'], ascending=[False,True])

def ordered_paths(folder, uid):
    paths = sorted(folder.glob('*.dcm'))
    if not paths: raise ValueError('No DICOM slices')
    hs = [header(p) for p in paths]
    if all(hasattr(h,'ImagePositionPatient') and hasattr(h,'ImageOrientationPatient') for h in hs):
        orient = np.asarray(hs[0].ImageOrientationPatient, dtype=float)
        normal = np.cross(orient[:3], orient[3:])
        if np.linalg.norm(normal)<0.9: raise ValueError('Invalid slice orientation')
        if not all(np.allclose(np.asarray(h.ImageOrientationPatient,float),orient,atol=0.05) for h in hs):
            raise ValueError('Mixed orientation within a series')
        positions = [float(np.dot(np.asarray(h.ImagePositionPatient,float),normal)) for h in hs]
    elif all(hasattr(h,'InstanceNumber') for h in hs):
        positions = [float(h.InstanceNumber) for h in hs]
        ISSUES.append(dict(study=uid, issue='InstanceNumber ordering fallback'))
    else:
        raise ValueError('Neither geometry nor complete InstanceNumber is available')
    order = np.argsort(positions, kind='stable')
    return [paths[i] for i in order]

def canonical_pixels(ds):
    arr = np.asarray(apply_modality_lut(ds.pixel_array, ds), dtype=np.float32)
    if arr.ndim!=2: raise ValueError(f'Expected single grayscale slice, received shape {arr.shape}')
    arr = np.nan_to_num(arr, nan=0, posinf=0, neginf=0)
    spacing = np.asarray(getattr(ds,'PixelSpacing',[1,1]),dtype=float)
    if not np.isfinite(spacing).all() or (spacing<=0).any(): spacing=np.ones(2)
    iop = getattr(ds,'ImageOrientationPatient',None)
    if iop is not None:
        col, row = np.array(iop[:3],float), np.array(iop[3:],float)
        if np.argmax(abs(col))>np.argmax(abs(row)):
            arr=arr.T; col,row=row,col; spacing=spacing[::-1]
        if col[np.argmax(abs(col))]<0: arr=np.fliplr(arr)
        if row[np.argmax(abs(row))]<0: arr=np.flipud(arr)
    if getattr(ds,'PhotometricInterpretation','MONOCHROME2')=='MONOCHROME1': arr=-arr
    return arr.copy(), spacing

def render_series(paths, size):
    idx = np.unique(np.linspace(0.25*(len(paths)-1),0.75*(len(paths)-1),min(SLICES_PER_PLANE,len(paths))).round().astype(int))
    decoded = [canonical_pixels(pydicom.dcmread(paths[i])) for i in idx]
    values = np.concatenate([a.ravel()[::max(1,a.size//10000)] for a,_ in decoded])
    lo,hi=np.percentile(values,[0.5,99.5])
    if hi<=lo: raise ValueError('Constant-intensity series')
    rendered=[]
    for arr,spacing in decoded:
        arr=np.uint8(np.clip((arr-lo)/(hi-lo),0,1)*255)
        ph,pw=np.array(arr.shape)*spacing
        nh,nw=max(1,round(size*ph/max(ph,pw))),max(1,round(size*pw/max(ph,pw)))
        im=Image.fromarray(arr).resize((nw,nh),Image.Resampling.BILINEAR)
        canvas=Image.new('L',(size,size)); canvas.paste(im,((size-nw)//2,(size-nh)//2))
        rendered.append(np.asarray(canvas))
    return rendered, idx/max(1,len(paths)-1)

def study_images(uid, descriptors, split='train', size=224):
    n=len(PLANES)*SLICES_PER_PLANE
    images=np.zeros((n,size,size),np.uint8); mask=np.zeros(n,bool); meta=np.zeros((n,6),np.float32)
    rows=descriptors[descriptors.StudyInstanceUID==uid]
    for pi,plane in enumerate(PLANES):
        choices=rank_series(rows[rows.Anatomical_Plane.astype(str).str.lower()==plane.lower()])
        for _,r in choices.iterrows():
            try:
                paths=ordered_paths(DATA/f'{split}_series'/uid/r.SeriesInstanceUID, uid)
                ims,pos=render_series(paths,size)
                sl=slice(pi*SLICES_PER_PLANE,pi*SLICES_PER_PLANE+len(ims))
                images[sl]=np.stack(ims); mask[sl]=True; meta[sl,pi]=1; meta[sl,3]=pos
                meta[sl,4]=float(r.Fluid_Sensitive) if pd.notna(r.Fluid_Sensitive) else 0
                meta[sl,5]=float(r.Fat_Suppression) if pd.notna(r.Fat_Suppression) else 0
                break
            except Exception as e:
                ISSUES.append(dict(study=uid, issue=f'{plane}: {type(e).__name__}: {str(e)[:160]}'))
    if not mask.any(): raise ValueError('No usable MRI series for this study')
    return images,mask,meta

uid=pilot.StudyInstanceUID.iloc[0]
ims,msk,meta=study_images(uid,series)
fig,axs=plt.subplots(1,3,figsize=(10,4))
for pi,ax in enumerate(axs):
    valid=np.flatnonzero(msk[pi*SLICES_PER_PLANE:(pi+1)*SLICES_PER_PLANE])+pi*SLICES_PER_PLANE
    if len(valid): ax.imshow(ims[valid[len(valid)//2]],cmap='gray')
    ax.set_title(PLANES[pi] if len(valid) else PLANES[pi]+' missing'); ax.axis('off')
plt.tight_layout(); fig.savefig(OUT/'preprocessing_preview.png',dpi=130); plt.show()
print('Decoded a real study:', int(msk.sum()), 'valid slice tokens; shape:', ims.shape)


## 4. Fixed development split
Use connected groups for exact normalized duplicate reports and usable patient identifiers. Patient identifiers, if present, are scoped to a scanner fingerprint to avoid conflating reused local IDs. Export group hashes, not patient metadata. This pilot does **not** establish independence between sites; the next experiment needs a larger audited reference set and a site/scanner holdout. We report scanner overlap explicitly.


In [ ]:
def digest(s): return hashlib.sha256(str(s).encode()).hexdigest()[:20]
parent=list(range(len(pilot)))
def root(i):
    while parent[i]!=i: parent[i]=parent[parent[i]]; i=parent[i]
    return i
def union(a,b): parent[root(b)]=root(a)
seen={}; scanner=[]
for i,r in pilot.iterrows():
    keys=[]
    report=str(r.get('Report','')) if pd.notna(r.get('Report',None)) else ''
    normalized=' '.join(report.lower().split())
    if normalized: keys.append('report:'+digest(normalized))
    paths=sorted((DATA/'train_series'/r.StudyInstanceUID).glob('*/*.dcm'))
    h=header(paths[0]) if paths else None
    fields=['Manufacturer','ManufacturerModelName','MagneticFieldStrength']
    sig='|'.join(str(getattr(h,k,'')) for k in fields)
    scanner.append(digest(sig) if sig.strip('|') else 'unknown')
    pid=str(getattr(h,'PatientID','')).strip()
    if pid and pid.lower() not in {'anonymous','anonymized','unknown','none','0','1'}:
        keys.append('patient:'+digest(sig+'|'+pid))
    for key in keys:
        if key in seen: union(i,seen[key])
        else: seen[key]=i
groups=np.array([digest(root(i)) for i in range(len(pilot))])
assert len(set(groups))>=4, 'Too few independent groups; revise split before training.'
tr,va=next(GroupShuffleSplit(n_splits=1,test_size=HOLDOUT_FRACTION,random_state=SEED).split(pilot,groups=groups))
pilot['split']='train'; pilot.loc[va,'split']='validation'
pilot['group']=groups; pilot['scanner_group']=scanner
assert set(pilot.loc[tr,'group']).isdisjoint(pilot.loc[va,'group'])
pilot[['StudyInstanceUID','split','group','scanner_group']].to_csv(OUT/'development_split.csv',index=False)
print('Train / validation studies:',len(tr),len(va))
print('Scanner groups shared across split:',len(set(pilot.loc[tr,'scanner_group']) & set(pilot.loc[va,'scanner_group'])))
print('Fixed diagnostic holdout only; not a site-independent performance estimate.')


In [ ]:
preflight=[]
for uid in tqdm(pilot.StudyInstanceUID,desc='Preflight MRI studies'):
    _,mask,_=study_images(uid,series,size=448)
    preflight.append(dict(StudyInstanceUID=uid,valid_slices=int(mask.sum())))
pd.DataFrame(preflight).to_csv(OUT/'preprocessing_coverage.csv',index=False)
assert all(r['valid_slices']==len(PLANES)*SLICES_PER_PLANE for r in preflight), (
    'The pilot expects every selected study to have all three planes; inspect preprocessing coverage.')
audit_status=dict(status='data_audit_complete',model=MODEL_ID,training_enabled=RUN_MEDGEMMA,
                  studies=len(train),organizer_labeled_studies=len(available),
                  pilot_train=len(tr),pilot_validation=len(va),slices_per_plane=SLICES_PER_PLANE)
(OUT/'audit_status.json').write_text(json.dumps(audit_status,indent=2))
pd.DataFrame(ISSUES,columns=['study','issue']).to_csv(OUT/'preprocessing_issues.csv',index=False)
print(json.dumps(audit_status,indent=2))


## 5. Optional MedGemma QLoRA pilot
The sections below are skipped unless `RUN_MEDGEMMA=True`. This is an adaptation of the official workflow, not a copy of its benchmark. It trains language attention adapters only, freezes the vision encoder and base weights, and computes loss only on known yes/no answers. It does not save large embedding or language-output layers as trainable modules.

Train/validation are split by study groups before creating study/condition examples. Missing labels produce no training example. The validation report is never used as an input or target. Numeric scores come from the relative next-token likelihoods of Yes and No, not numbers invented by generated text; these are still uncalibrated ranking scores.


In [ ]:
if RUN_MEDGEMMA:
    assert torch.cuda.is_available(), 'Select a suitable GPU before model training.'
    assert torch.cuda.get_device_capability(0)[0]>=8, (
        'This recipe requires native BF16 (Ampere or newer). T4 adaptation is not verified.')
    assert torch.cuda.get_device_properties(0).total_memory/2**30>=37, (
        'This conservative pilot requires about 40 GB on one GPU. Two T4s do not pool memory.')
    subprocess.check_call([sys.executable,'-m','pip','install','--quiet',
                           'transformers>=4.56,<5','peft>=0.17,<1',
                           'accelerate>=1.10,<2','bitsandbytes>=0.47,<1','sentencepiece'])
    from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    from importlib.metadata import version
    access_token=os.environ.get('HF_TOKEN')
    if not access_token:
        try:
            from kaggle_secrets import UserSecretsClient
            access_token=UserSecretsClient().get_secret('HF_TOKEN')
        except Exception:
            raise RuntimeError('Set the HF_TOKEN secret after obtaining official MedGemma access.') from None
    processor=AutoProcessor.from_pretrained(MODEL_ID,token=access_token)
    processor.tokenizer.padding_side='right'
    quant=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_use_double_quant=True,
                            bnb_4bit_quant_type='nf4',bnb_4bit_compute_dtype=torch.bfloat16)
    model=AutoModelForImageTextToText.from_pretrained(
        MODEL_ID,token=access_token,quantization_config=quant,
        torch_dtype=torch.bfloat16,device_map={'':0},attn_implementation='eager')
    del access_token
    model=prepare_model_for_kbit_training(model,use_gradient_checkpointing=True,
                                        gradient_checkpointing_kwargs={'use_reentrant':False})
    targets=[n for n,m in model.named_modules()
             if 'language_model' in n and n.endswith(('.q_proj','.k_proj','.v_proj','.o_proj'))]
    assert targets, 'Inspect this model version before selecting LoRA modules.'
    model=get_peft_model(model,LoraConfig(r=LORA_RANK,lora_alpha=2*LORA_RANK,
                         lora_dropout=0.05,bias='none',target_modules=targets,task_type='CAUSAL_LM'))
    model.config.use_cache=False
    model.print_trainable_parameters()
    versions={p:version(p) for p in ['torch','transformers','peft','bitsandbytes','pydicom']}
    config=dict(model=MODEL_ID,resolved_revision=getattr(model.config,'_commit_hash',None),
                precision='nf4_with_bf16_autocast',
                seed=SEED,steps=MAX_STEPS,accumulation=ACCUMULATION,lora_rank=LORA_RANK,
                learning_rate=LEARNING_RATE,slices_per_plane=SLICES_PER_PLANE,versions=versions)
    (OUT/'training_config.json').write_text(json.dumps(config,indent=2))
else:
    print('Skipped model download and QLoRA setup. See hardware and access notes above.')


In [ ]:
DEFINITIONS = {
 'ACL':'high-grade partial (>50%) or complete anterior cruciate ligament tear; exclude isolated low-grade sprain or degeneration',
 'MCL':'high-grade acute partial or complete medial collateral ligament tear; exclude low-grade sprain or chronic healed injury',
 'Medial Meniscus':'definite medial meniscus tear with surface-reaching signal on at least two images or abnormal morphology; exclude isolated intrameniscal degeneration',
 'Lateral Meniscus':'definite lateral meniscus tear with surface-reaching signal on at least two images or abnormal morphology; exclude isolated intrameniscal degeneration',
 'Medial OA':'medial compartment cartilage loss of high grade (>50% thickness) over a moderate or large area (at least 1 cm)',
 'Lateral OA':'lateral compartment cartilage loss of high grade (>50% thickness) over a moderate or large area (at least 1 cm)',
 'PF OA':'patellofemoral cartilage loss of high grade (>50% thickness) over a moderate or large area (at least 1 cm)',
 'Effusion':'moderate or large joint effusion',
 'Synovitis':'synovitis with synovial thickening or inflammation',
 "Baker's":'moderate or large popliteal (Baker) cyst',
 'Contusion':'traumatic bone marrow edema (bone contusion) without a fracture line; exclude degenerative edema',
 'Fracture':'an acute fracture',
}
IMAGE_CACHE={}
def messages_for(uid,label,answer=None):
    if uid not in IMAGE_CACHE:
        arr,mask,metadata=study_images(uid,series,size=448)
        IMAGE_CACHE[uid]=[(Image.fromarray(arr[i]).convert('RGB'),metadata[i]) for i in np.flatnonzero(mask)]
    content=[dict(type='text',text='These are ordered sampled slices from one knee MRI study. Some anatomy may not be visible.')]
    for im,m in IMAGE_CACHE[uid]:
        content.extend([dict(type='text',text=f'{PLANES[int(np.argmax(m[:3]))]}, slice position {m[3]:.2f}, fluid sensitive {int(m[4])}, fat suppression {int(m[5])}.'),dict(type='image',image=im)])
    content.append(dict(type='text',text=f'Is there {DEFINITIONS[label]}? Answer with only Yes or No.'))
    messages=[dict(role='user',content=content)]
    if answer is not None: messages.append(dict(role='assistant',content=[dict(type='text',text=answer)]))
    return messages

def encode_question(uid,label,answer=None):
    prompt=messages_for(uid,label)
    p=processor.apply_chat_template(prompt,add_generation_prompt=True,tokenize=True,
                                    return_dict=True,return_tensors='pt')
    if answer is None: return p
    batch=processor.apply_chat_template(messages_for(uid,label,answer),add_generation_prompt=False,
                                        tokenize=True,return_dict=True,return_tensors='pt')
    prefix=p['input_ids'].shape[1]
    assert torch.equal(p['input_ids'],batch['input_ids'][:,:prefix]), 'Unexpected chat template boundary.'
    batch['labels']=batch['input_ids'].clone()
    batch['labels'][:,:prefix]=-100
    assert (batch['labels']!=-100).sum()>0
    return batch

def on_gpu(batch):
    return {k:v.to('cuda:0',dtype=torch.bfloat16) if v.is_floating_point() else v.to('cuda:0')
            for k,v in batch.items()}

@torch.inference_mode()
def predict_scores(frame):
    model.eval()
    rows=[]
    for _,row in tqdm(frame.iterrows(),total=len(frame),desc='Teacher scores'):
        result={'StudyInstanceUID':row.StudyInstanceUID,'split':row['split']}
        for label in LABELS:
            batch=on_gpu(encode_question(row.StudyInstanceUID,label))
            with torch.autocast('cuda',dtype=torch.bfloat16):
                logits=model(**batch,use_cache=False,logits_to_keep=1).logits[0,-1].float()
            pair=logits[answer_ids]
            assert torch.isfinite(pair).all(), 'Non-finite teacher scores.'
            result[label]=float(pair.softmax(0)[1].cpu())
            del logits,pair,batch
        rows.append(result)
    return pd.DataFrame(rows)

def score_auc(predictions,tag):
    aligned=pilot.set_index('StudyInstanceUID').loc[predictions.StudyInstanceUID]
    rows=[]
    for label in LABELS:
        y=aligned[label].to_numpy(dtype=float); p=predictions[label].to_numpy(dtype=float)
        known=np.isfinite(y); auc=roc_auc_score(y[known],p[known]) if len(np.unique(y[known]))==2 else np.nan
        rows.append(dict(model=tag,label=label,n=int(known.sum()),auc=auc))
    return pd.DataFrame(rows)


## 6. Compare before and after adaptation
The held-out studies are scored before training and again after a fixed number of updates. They are not used for gradient updates or checkpoint selection. Undefined AUCs stay missing. Averages over evaluable labels are diagnostic, not the official 12-label competition metric. The yes/no format is an experimental approximation; the small slice sample cannot establish every diagnostic criterion above.


In [ ]:
if RUN_MEDGEMMA:
    # Verify the label text is a single token in this exact tokenizer/template.
    answer_ids=[]
    for answer in ['No','Yes']:
        ids=processor.tokenizer.encode(answer,add_special_tokens=False)
        assert len(ids)==1, 'Use complete answer-sequence likelihoods for this tokenizer.'
        answer_ids.append(ids[0])
    first=pilot.iloc[tr[0]]
    for answer,token_id in zip(['No','Yes'],answer_ids):
        check=encode_question(first.StudyInstanceUID,LABELS[0],answer)
        active=check['labels'][check['labels']!=-100]
        assert active[0].item()==token_id, 'Answer-token boundary differs from isolated tokenization.'
    validation=pilot.iloc[va]
    before=predict_scores(validation)
    before.to_csv(OUT/'validation_before.csv',index=False)
    log_metric_event(0,{'validation/mean_auc':float(score_auc(before,'base').auc.mean())})
    examples=[(r.StudyInstanceUID,label,'Yes' if r[label]==1 else 'No')
              for _,r in pilot.iloc[tr].iterrows() for label in LABELS if pd.notna(r[label])]
    assert examples
    optimizer=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=LEARNING_RATE)
    model.train(); optimizer.zero_grad(set_to_none=True)
    history=[]; rng=random.Random(SEED)
    torch.cuda.reset_peak_memory_stats(); started=time.time()
    for step in tqdm(range(MAX_STEPS),desc='QLoRA updates'):
        losses=[]
        for micro in range(ACCUMULATION):
            uid,label,answer=rng.choice(examples)
            batch=on_gpu(encode_question(uid,label,answer))
            with torch.autocast('cuda',dtype=torch.bfloat16):
                loss=model(**batch,use_cache=False).loss
            assert torch.isfinite(loss), 'Non-finite loss: stop and inspect precision/data.'
            (loss/ACCUMULATION).backward()
            losses.append(float(loss.detach().cpu()))
            del batch,loss
        grad_norm=torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad],1.0)
        assert torch.isfinite(grad_norm), 'Non-finite gradients.'
        optimizer.step(); optimizer.zero_grad(set_to_none=True)
        history.append(dict(step=step+1,loss=float(np.mean(losses))))
        pd.DataFrame(history).to_csv(OUT/'training_history.csv',index=False)
        log_metric_event(step+1,{'train/loss':history[-1]['loss'],
            'train/gradient_norm':float(grad_norm.detach().cpu()),
            'train/learning_rate':LEARNING_RATE,
            'gpu/allocated_gib':torch.cuda.memory_allocated()/2**30})
    seconds=time.time()-started
    training_peak_gib=torch.cuda.max_memory_allocated()/2**30
    pd.DataFrame(history).to_csv(OUT/'training_history.csv',index=False)
    model.save_pretrained(OUT/'teacher_adapter',safe_serialization=True)
    processor.save_pretrained(OUT/'teacher_adapter')
    after=predict_scores(validation)
    after.to_csv(OUT/'validation_after.csv',index=False)
    metrics=pd.concat([score_auc(before,'base'),score_auc(after,'adapted')],ignore_index=True)
    metrics.to_csv(OUT/'validation_metrics.csv',index=False)
    log_metric_event(MAX_STEPS,{'validation/mean_auc':float(score_auc(after,'adapted').auc.mean())})
    display(metrics.pivot(index='label',columns='model',values='auc'))
    print('Training seconds:',round(seconds),'peak allocated GiB:',round(training_peak_gib,2))
    # These scores are for training studies only; they are in-sample, not OOF.
    teacher_targets=predict_scores(pilot.iloc[tr])
    teacher_targets.to_csv(OUT/'teacher_targets_train_in_sample.csv',index=False)
    (OUT/'pilot_status.json').write_text(json.dumps(dict(status='teacher_pilot_complete',
        training_seconds=seconds,training_studies=len(tr),validation_studies=len(va),
        peak_training_allocated_gib=training_peak_gib,
        teacher_scores='uncalibrated_yes_no_ranking_scores',student_trained=False),indent=2))
else:
    print('Training and teacher scoring skipped. The CPU audit is the only executed experiment.')


## 7. Student handoff and next experiment
1. Review the label audit. Build a substantially larger reference set or audit report-derived weak labels. The original radiology reports are useful training supervision, but unavailable at test time. Unknown/uncertain findings must remain unknown.
2. Expand MRI sampling using Google's volumetric example, preserving plane and slice information. Check that cropping and resizing preserve small structures. Benchmark more than a few labeled studies before choosing the teacher.
3. Train the teacher on the training folds only. If it improves held-out discrimination, export logits or audited probability targets for those folds. The pilot's `teacher_targets_train_in_sample.csv` is explicitly in-sample and excludes validation studies; it is not an out-of-fold evaluation artifact.
4. Train a small multi-plane image classifier with known-label binary cross entropy plus a weighted teacher-target loss. Evaluate the same student trained without distillation as a control. Do not use held-out labels or reports to train either model.
5. Save the student weights and dependencies as authorized Kaggle inputs. Build a separate inference notebook that runs offline, respects the competition runtime limit, and writes `submission.csv`.

MedGemma is a candidate, not a proven best knee classifier. Google recommends MedSigLIP for image-only classification, so that is a useful comparison. Retain Google license notices for adapters and distilled derivatives. The competition generally permits external models, but the host retains eligibility decisions; no MedGemma-specific host ruling has been established here.

## Sources
- [Competition data](https://www.kaggle.com/competitions/rsna-knee-abnormality-detection/data)
- [Competition rules — external models and winner obligations](https://www.kaggle.com/competitions/rsna-knee-abnormality-detection/rules)
- [Organizer label definitions](https://www.kaggle.com/competitions/rsna-knee-abnormality-detection/discussion/733343)
- [Google MedGemma 1.5 model card](https://developers.google.com/health-ai-developer-foundations/medgemma/model-card)
- [Google fine-tuning example — BF16 / 40 GB requirement](https://github.com/Google-Health/medgemma/blob/main/notebooks/fine_tune_with_hugging_face.ipynb)
- [Google volumetric preprocessing example](https://github.com/Google-Health/medgemma/blob/main/notebooks/high_dimensional_ct_hugging_face.ipynb)
- [HAI-DEF terms — modification and distilled derivatives](https://developers.google.com/health-ai-developer-foundations/terms)
- [Meta fastMRI](https://ai.meta.com/research/impact/fastmri/)
